# TMDB Movies Per Capita Analysis

This notebook analyzes movies per capita by country from the TMDB dataset.
It allows filtering by year and continent.
Data is collected from TMDB API and acquired from Kaggle https://www.kaggle.com/datasets/asaniczka/tmdb-movies-dataset-2023-930k-movies/data

In [ ]:
import os
import sys
import warnings

import pandas as pd
import polars as pl
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib as mpl
from matplotlib.collections import PatchCollection
from matplotlib.patches import Polygon
from mpl_toolkits.basemap import Basemap
import numpy as np
from typing import Dict, List, Optional

from mapy.config import Continent

from mapy.config import CONTINENT_MAPPING, COUNTRY_POPULATION_DATA
from mapy.paths import DATA_DIR, OUTPUT_DIR

warnings.filterwarnings('ignore')

# Set style for visualizations
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 8)

In [ ]:
def load_tmdb_data(filepath: str) -> pl.DataFrame:
    '''
    Load TMDB movie dataset from CSV file.
    '''
    df = pl.read_csv(filepath)
    return df


def get_country_continent(country_name: str) -> Optional[str]:
    '''
    Map a country name to its continent.
    '''
    for continent, data in CONTINENT_MAPPING.items():
        if country_name in data['countries']:
            return continent
    return None


def extract_countries_from_production(production_str: Optional[str]) -> List[str]:
    '''
    Extract country names from production_countries string.
    Expected format: "Country1, Country2, Country3"
    '''
    if production_str is None or production_str == '':
        return []
    return [c.strip() for c in str(production_str).split(',')]


def process_tmdb_data(
    df: pl.DataFrame,
    year: int,
    continent: str = 'Europe'
) -> pl.DataFrame:
    '''
    Process TMDB data to calculate movies per capita by country.

    Args:
        df: TMDB dataframe
        year: Year to filter by
        continent: Continent to filter by (default: Europe)

    Returns:
        DataFrame with countries and their movies per capita
    '''
    # Convert release_date to datetime and extract year
    df = df.with_columns(
        pl.col('release_date').str.to_date(strict=False).dt.year().alias('release_year')
    )

    # Filter by year
    df_year = df.filter(pl.col('release_year') == year)

    # Extract countries and expand rows
    expanded_data = []
    for row in df_year.iter_rows(named=True):
        countries = extract_countries_from_production(row['production_countries'])
        for country in countries:
            expanded_data.append({
                'country': country,
                'movie_id': row['id']
            })

    if not expanded_data:
        return pl.DataFrame()

    df_countries = pl.DataFrame(expanded_data)

    # Add continent mapping
    df_countries = df_countries.with_columns(
        pl.col('country').map_elements(
            get_country_continent,
            return_dtype=pl.Utf8
        ).alias('continent')
    )

    # Filter by continent
    df_countries = df_countries.filter(pl.col('continent') == continent)

    # Count movies per country
    movies_per_country = (
        df_countries
        .group_by('country')
        .agg(pl.count('movie_id').alias('movie_count'))
    )

    # Add population data
    movies_per_country = movies_per_country.with_columns(
        pl.col('country').map_elements(
            lambda x: COUNTRY_POPULATION_DATA.get(x),
            return_dtype=pl.Float64
        ).alias('population')
    )

    # Calculate movies per capita (per million people)
    movies_per_country = movies_per_country.with_columns(
        (pl.col('movie_count') / pl.col('population')).alias('movies_per_capita')
    )

    # Remove countries with missing population data
    movies_per_country = movies_per_country.drop_nulls()

    # Sort by movies per capita
    movies_per_country = movies_per_country.sort('movies_per_capita', descending=True)

    return movies_per_country

In [ ]:
# Load data
df_tmdb = load_tmdb_data(os.path.join(DATA_DIR, 'TMDB_movie_dataset_v11.csv'))

print(f'Total movies in dataset: {len(df_tmdb):,}')
print('\nSample data:')
print(df_tmdb.select(['title', 'release_date', 'production_countries']).head(10))

In [ ]:
# Parameters for analysis
YEAR = 2025
CONTINENT = 'Europe'
COUNTRIES_TO_EXCLUDE = ['Cyprus']

# Process data
result_df = process_tmdb_data(
    df_tmdb,
    year=YEAR,
    continent=CONTINENT
)

print('Countries before exclusion:', sorted(result_df['country'].to_list()))

# Exclude specified countries
for country in COUNTRIES_TO_EXCLUDE:
    result_df = result_df.filter(pl.col('country') != country)
print('Countries after exclusion:', sorted(result_df['country'].to_list()))

print(f'\nAnalysis: {CONTINENT} - Year {YEAR}')
print(f'Countries found: {len(result_df)}')
print('\nTop 15 countries by movies per capita:')
print(result_df.select(['country', 'movie_count', 'population', 'movies_per_capita']).head(15))

## Bar Chart Visualization

In [ ]:
# Bar chart of top 20 countries
top_n = 20
top_countries = result_df.head(top_n)

# Convert to pandas for easy plotting
top_countries_pd = top_countries.to_pandas()

fig, ax = plt.subplots(figsize=(12, 8))

bars = ax.barh(
    range(len(top_countries_pd)),
    top_countries_pd['movies_per_capita'],
    color='steelblue'
)

ax.set_yticks(range(len(top_countries_pd)))
ax.set_yticklabels(top_countries_pd['country'])
ax.set_xlabel('Movies per Capita (per million people)', fontsize=12)
ax.set_title(
    f'Top {top_n} {CONTINENT} Countries by Movies Per Capita ({YEAR})',
    fontsize=14,
    fontweight='bold'
)

# Add value labels on bars
for i, row in enumerate(top_countries_pd.itertuples()):
    ax.text(
        row.movies_per_capita,
        i,
        f' {row.movies_per_capita:.4f}',
        va='center',
        fontsize=9
    )

plt.tight_layout()
plt.show()

print(f'Chart shows top {top_n} countries by movies per capita')

## Map Visualization

In [ ]:
# Map Visualization - Color entire countries by movies per capita

# Configuration parameters
MAP_DPI = 300
MAP_COLORMAP = 'YlOrRd'  # Options: 'YlOrRd', 'RdYlGn_r', 'Oranges', 'Reds', etc.
COLORMAP_START = 0.3  # Start from this point in colormap (0.0-1.0)
COLORMAP_END = 1.0    # End at this point in colormap (0.0-1.0)

# Label position shifts (in map projection units)
LABEL_SHIFTS = {
    'Russia': (-400000, 0),      # Shift west
    'Ukraine': (-300000, +1000),     # Shift west
    'Turkey': (-400000, 0),      # Shift west
    'Norway': (0, -100000),      # Shift south slightly
    'Sweden': (0, -100000),      # Shift south slightly
    'Albania': (+50000, 0),    # Shift east
    'Montenegro': (+50000, 0),    # Shift east
}

# Map configuration
SHAPE_FILE = os.path.join(DATA_DIR, 'natural_earth/ne_10m_admin_0_countries')
RESOLUTION = 'i'
FIGSIZE = (20, 16)
N_COLORS = 9

# Get country coordinates for text labels
country_coords_df = pl.read_csv(os.path.join(DATA_DIR, 'world_country_lon_lat.csv'))

# Convert result_df to pandas
result_pd = result_df.to_pandas()

# Join coordinates for text labels
coords_pd = country_coords_df.to_pandas()
result_pd = result_pd.merge(
    coords_pd[['name', 'lat', 'lon']], 
    left_on='country', 
    right_on='name', 
    how='left'
)

# Set up color bins
movies_per_capita_values = result_pd['movies_per_capita']
cm = plt.get_cmap(MAP_COLORMAP)
# Generate colors with configurable range
scheme = np.array([cm(COLORMAP_START + (i / N_COLORS) * (COLORMAP_END - COLORMAP_START)) 
                   for i in range(N_COLORS)])
# Create N_COLORS+1 bin edges for N_COLORS bins
bins = np.linspace(movies_per_capita_values.min(), movies_per_capita_values.max(), N_COLORS + 1)
result_pd['bin'] = np.digitize(movies_per_capita_values, bins[:-1]) - 1
result_pd['bin'] = result_pd['bin'].clip(0, N_COLORS - 1)

# Get region configuration
if CONTINENT == 'Europe':
    region = Continent.europe
else:
    region = Continent.world

# Create figure
fig = plt.figure(figsize=FIGSIZE)
ax = fig.add_subplot(111, frame_on=False)
fig.suptitle(f'Movie Counts per Million People in {CONTINENT} in {YEAR}', 
             fontsize=22, y=0.96, fontweight='bold')

# Create Basemap
if CONTINENT == 'Europe':
    m = Basemap(
        llcrnrlon=region['llcrnrlon'],
        llcrnrlat=region['llcrnrlat'] - 1.0,
        urcrnrlon=region['urcrnrlon'] - 20.0,
        urcrnrlat=region['urcrnrlat'],
        lat_0=region['lat_0'],
        lon_0=region['lon_0'],
        lat_1=region['lat_1'],
        lat_2=region['lat_2'],
        projection='lcc',
        resolution=RESOLUTION,
        area_thresh=1,
    )
else:
    m = Basemap(
        llcrnrlon=region['llcrnrlon'],
        llcrnrlat=region['llcrnrlat'],
        urcrnrlon=region['urcrnrlon'],
        urcrnrlat=region['urcrnrlat'],
        projection='mill',
        resolution=RESOLUTION,
        area_thresh=1,
    )

# Draw map boundary
m.drawmapboundary(fill_color='lightblue', linewidth=0.5)

# Comprehensive name mapping (shapefile name -> TMDB country name)
name_mapping = {
    'United Kingdom': 'United Kingdom',
    'France': 'France',
    'Germany': 'Germany',
    'Italy': 'Italy',
    'Spain': 'Spain',
    'Netherlands': 'Netherlands',
    'Belgium': 'Belgium',
    'Switzerland': 'Switzerland',
    'Austria': 'Austria',
    'Sweden': 'Sweden',
    'Norway': 'Norway',
    'Denmark': 'Denmark',
    'Finland': 'Finland',
    'Poland': 'Poland',
    'Czechia': 'Czech Republic',
    'Czech Republic': 'Czech Republic',
    'Hungary': 'Hungary',
    'Romania': 'Romania',
    'Bulgaria': 'Bulgaria',
    'Greece': 'Greece',
    'Portugal': 'Portugal',
    'Ireland': 'Ireland',
    'Iceland': 'Iceland',
    'Luxembourg': 'Luxembourg',
    'Slovenia': 'Slovenia',
    'Croatia': 'Croatia',
    'Serbia': 'Serbia',
    'Slovakia': 'Slovakia',
    'Estonia': 'Estonia',
    'Latvia': 'Latvia',
    'Lithuania': 'Lithuania',
    'Bosnia and Herzegovina': 'Bosnia and Herzegovina',
    'Bosnia and Herz.': 'Bosnia and Herzegovina',
    'Albania': 'Albania',
    'Cyprus': 'Cyprus',
    'Malta': 'Malta',
    'Montenegro': 'Montenegro',
    'Ukraine': 'Ukraine',
    'Belarus': 'Belarus',
    'Russia': 'Russia',
    'Turkey': 'Turkey',
    # Additional mappings for disputed/variant names
    'Macedonia': None,  # North Macedonia - not in our 2020 data
    'North Macedonia': None,  # Not in our 2020 data
    'Kosovo': None,  # Not in our 2020 data
    'Crimea': 'Ukraine',  # Map Crimea to Ukraine
    'Republic of Serbia': 'Serbia',
}

# Create lookup dictionary
country_data = {}
for _, row in result_pd.iterrows():
    country_data[row['country']] = {
        'bin': int(row['bin']),
        'value': row['movies_per_capita']
    }

# Read shapefile and color countries
m.readshapefile(SHAPE_FILE, 'units', color='#444444', linewidth=0.2)

country_names_in_df = np.array(result_pd['country'])
for info, shape in zip(m.units_info, m.units):
    ne_name = info['NAME_EN'].strip()
    
    # Try to find matching country
    tmdb_name = name_mapping.get(ne_name, ne_name)  # Use shapefile name as fallback
    
    # Skip if explicitly mapped to None (not in our data)
    if tmdb_name is None:
        continue
    
    if tmdb_name in country_data:
        # Get color from bin
        color = scheme[country_data[tmdb_name]['bin']]
        patches = [Polygon(np.array(shape))]
        pc = PatchCollection(patches)
        pc.set_facecolor(color)
        ax.add_collection(pc)

# Add text labels for values with position shifts
for _, row in result_pd.iterrows():
    if pd.notna(row['lat']) and pd.notna(row['lon']):
        x, y = m(row['lon'], row['lat'])
        
        # Apply shift if country is in LABEL_SHIFTS
        if row['country'] in LABEL_SHIFTS:
            shift_x, shift_y = LABEL_SHIFTS[row['country']]
            x += shift_x
            y += shift_y
        
        value_text = f"{row['movies_per_capita']:.1f}"
        plt.text(x, y, value_text, 
                ha='center', va='center',
                fontsize=9, fontweight='bold',
                color='darkred',
                bbox=dict(facecolor='white', alpha=0.7, 
                         edgecolor='none', boxstyle='round,pad=0.2'))

# Draw color legend at top below title
ax_legend = fig.add_axes([0.25, 0.91, 0.5, 0.015], zorder=3)
cmap = mpl.colors.ListedColormap(scheme)
norm = mpl.colors.BoundaryNorm(bins, N_COLORS)
cb = mpl.colorbar.ColorbarBase(
    ax_legend,
    cmap=cmap,
    norm=norm,
    boundaries=bins,
    ticks=bins,
    orientation='horizontal'
)
cb.ax.set_xticklabels([f'{b:.1f}' for b in bins], fontsize=10)
cb.set_label('Movies per Million People', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/tmdb_movies_per_capita_{CONTINENT.lower()}_{YEAR}.png', 
            dpi=MAP_DPI, bbox_inches='tight')
print(f'Saved map for {CONTINENT} {YEAR} as data/output/tmdb_movies_per_capita_{CONTINENT.lower()}_{YEAR}.png (DPI: {MAP_DPI})')
plt.show()

## Summary Statistics

In [ ]:
# Summary statistics
print(f'\n=== Summary Statistics for {CONTINENT} ({YEAR}) ===')
print(f'\nTotal countries analyzed: {len(result_df)}')
print(f'Total movies produced: {result_df["movie_count"].sum():.0f}')
print(f'Total population: {result_df["population"].sum():.1f} million')
print('\nMovies per capita statistics:')
print(f'  Mean: {result_df["movies_per_capita"].mean():.6f}')
print(f'  Median: {result_df["movies_per_capita"].median():.6f}')
print(f'  Std Dev: {result_df["movies_per_capita"].std():.6f}')
print(f'  Min: {result_df["movies_per_capita"].min():.6f} ({result_df[-1, "country"]})')
print(f'  Max: {result_df["movies_per_capita"].max():.6f} ({result_df[0, "country"]})')
print('\nTop 5 countries:')
print(result_df.select(['country', 'movie_count', 'movies_per_capita']).head(5))